## Modelos Supervisados Avanzados

In [29]:
import pandas as pd
import numpy as np
import mlflow

from sklearn.ensemble import StackingClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import VotingClassifier, StackingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


In [3]:
import dagshub
dagshub.init(repo_owner='davidrodriguez2712',
             repo_name='ml-2-modulo',
             mlflow=True)

Accessing as davidrodriguez2712

Initialized MLflow to track repo "davidrodriguez2712/ml-2-modulo"

Repository davidrodriguez2712/ml-2-modulo initialized!

## Baseline

In [5]:
df = pd.read_csv("../data/raw/hotel_bookings.csv")
df

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,City Hotel,0,23,2017,August,35,30,2,5,2,...,No Deposit,394.0,NaN,0,Transient,96.14,0,0,Check-Out,2017-09-06
119386,City Hotel,0,102,2017,August,35,31,2,5,3,...,No Deposit,9.0,NaN,0,Transient,225.43,0,2,Check-Out,2017-09-07
119387,City Hotel,0,34,2017,August,35,31,2,5,2,...,No Deposit,9.0,NaN,0,Transient,157.71,0,4,Check-Out,2017-09-07
119388,City Hotel,0,109,2017,August,35,31,2,5,2,...,No Deposit,89.0,NaN,0,Transient,104.40,0,0,Check-Out,2017-09-07


In [39]:
FEATURES = ["lead_time", "stays_in_week_nights", "children", "adr", "booking_changes"]


In [40]:
# FEATURES = ["lead_time", "stays_in_week_nights", "children", "adr"]
X, y = df[FEATURES], df["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 100)

In [41]:
X_train.head()

,lead_time,stays_in_week_nights,children,adr,booking_changes
10716,180,4,0.0,55.8,3
111508,112,1,0.0,126.0,0
57246,440,2,0.0,62.0,0
2498,47,4,0.0,76.8,0
58175,232,2,0.0,98.1,0


In [8]:
mlflow.set_tracking_uri("https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow")


In [9]:
mlflow.create_experiment("DSRP - Booking Clase 14 Julio")

'1'

In [10]:
mlflow.set_experiment("DSRP - Booking Clase 14 Julio")

<Experiment: artifact_location='mlflow-artifacts:/6c8641e9c23f4c40bc83869385c7ffeb', creation_time=1775924465349, experiment_id='1', last_update_time=1775924465349, lifecycle_stage='active', name='DSRP - Booking Clase 14 Julio', tags={}, trace_location=None, workspace='default'>

### Dummy Classifier

In [42]:
mlflow.autolog()

with mlflow.start_run(run_name = "Baseline - Dummy Classifier - Con Métricas") as run:
    algorithm = DummyClassifier(strategy = "most_frequent")
    algorithm.fit(X_train, y_train)

    predictions = algorithm.predict(X_test)
    _accuracy_score = accuracy_score(y_test, predictions)
    _f1_score = f1_score(y_test, predictions)

    mlflow.log_metrics(
        {
            "accuracy": _accuracy_score,
            "f1": _f1_score
        }
    )

2026/04/12 11:09:58 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.
2026/04/12 11:09:58 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/04/12 11:09:58 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/04/12 11:10:00 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values.

🏃 View run Baseline - Dummy Classifier - Con Métricas at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2/runs/305f1b95e3394b82b7afcbce85b0875b
🧪 View experiment at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2


### Regresión Logística

In [43]:
with mlflow.start_run(run_name = "Regresión Logística") as run:
    algorithm = LogisticRegression()
    pipeline = Pipeline(
        steps = [
            ("imputer", SimpleImputer(strategy = "mean")),
            ("reg_logistica", algorithm)
        ]
    )
    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    _accuracy_score = accuracy_score(y_test, predictions)
    _f1_score = f1_score(y_test, predictions)

    mlflow.log_metrics(
        {
            "accuracy": _accuracy_score,
            "f1": _f1_score
        }
    )
    

2026/04/12 11:10:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/12 11:10:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de

🏃 View run Regresión Logística at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2/runs/5bd6b3513a9b4ef0bb97fe3471319610
🧪 View experiment at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2


### Emsemble : Bagging 1

In [44]:
with mlflow.start_run(run_name = "Bagging") as run:
    algorithm = BaggingClassifier()
    pipeline = Pipeline(
        steps = [
            ("imputer", SimpleImputer(strategy = "mean")),
            ("bagging", algorithm)
        ]
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    _accuracy_score = accuracy_score(y_test, predictions)
    _f1_score = f1_score(y_test, predictions)
    
    mlflow.log_metrics(
        {
            "accuracy": _accuracy_score,
            "f1": _f1_score
        }
    )

2026/04/12 11:10:45 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/12 11:10:46 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de

🏃 View run Bagging at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2/runs/67d612f4d18046228fc9c4d0ed6a9432
🧪 View experiment at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2


### RandomForest

In [45]:
with mlflow.start_run(run_name = "Random Forest") as run:
    algorithm = RandomForestClassifier()
    pipeline = Pipeline(
        steps = [
            ("imputer", SimpleImputer(strategy = "mean")),
            ("rf", algorithm)
        ]
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    _accuracy_score = accuracy_score(y_test, predictions)
    _f1_score = f1_score(y_test, predictions)
    
    mlflow.log_metrics(
        {
            "accuracy": _accuracy_score,
            "f1": _f1_score
        }
    )

2026/04/12 11:11:20 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/12 11:11:24 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de

🏃 View run Random Forest at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2/runs/556a809f39c94417877a7e20bab37c85
🧪 View experiment at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2


### Ensamble 3: XGBoost


In [46]:
with mlflow.start_run(run_name = "XGBoost") as run:
    algorithm = XGBClassifier()
    pipeline = Pipeline(
        steps = [
            ("imputer", SimpleImputer(strategy = "mean")),
            ("xgboost", algorithm)
        ]
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    _accuracy_score = accuracy_score(y_test, predictions)
    _f1_score = f1_score(y_test, predictions)
    
    mlflow.log_metrics(
        {
            "accuracy": _accuracy_score,
            "f1": _f1_score
        }
    )

2026/04/12 11:19:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/12 11:19:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de

🏃 View run XGBoost at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2/runs/dfdc80706d9d4ab2b07196d39d065f6d
🧪 View experiment at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2


### Ensamble 4

In [47]:
with mlflow.start_run(run_name = "LGBM") as run:
    algorithm = LGBMClassifier()
    pipeline = Pipeline(
        steps = [
            ("imputer", SimpleImputer(strategy = "mean")),
            ("lgbm", algorithm)
        ]
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    _accuracy_score = accuracy_score(y_test, predictions)
    _f1_score = f1_score(y_test, predictions)
    
    mlflow.log_metrics(
        {
            "accuracy": _accuracy_score,
            "f1": _f1_score
        }
    )

2026/04/12 11:20:15 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


[LightGBM] [Info] Number of positive: 33167, number of negative: 56375
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000739 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 555
[LightGBM] [Info] Number of data points in the train set: 89542, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.370407 -> initscore=-0.530470
[LightGBM] [Info] Start training from score -0.530470


2026/04/12 11:20:16 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/12 11:20:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution

🏃 View run LGBM at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2/runs/3fd2e7208a18458ba96009c24c84afd1
🧪 View experiment at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2


### Catboost

In [48]:
with mlflow.start_run(run_name = "Catboost") as run:
    algorithm = CatBoostClassifier()
    pipeline = Pipeline(
        steps = [
            ("imputer", SimpleImputer(strategy = "mean")),
            ("catboost", algorithm)
        ]
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    _accuracy_score = accuracy_score(y_test, predictions)
    _f1_score = f1_score(y_test, predictions)
    
    mlflow.log_metrics(
        {
            "accuracy": _accuracy_score,
            "f1": _f1_score
        }
    )

2026/04/12 11:20:43 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Learning rate set to 0.070218
0:	learn: 0.6738107	total: 5.15ms	remaining: 5.15s
1:	learn: 0.6577171	total: 8.39ms	remaining: 4.19s
2:	learn: 0.6435684	total: 12.2ms	remaining: 4.07s
3:	learn: 0.6321459	total: 15.6ms	remaining: 3.9s
4:	learn: 0.6222569	total: 18.8ms	remaining: 3.75s
5:	learn: 0.6135038	total: 22.2ms	remaining: 3.67s
6:	learn: 0.6061515	total: 25.5ms	remaining: 3.61s
7:	learn: 0.5998937	total: 29.1ms	remaining: 3.6s
8:	learn: 0.5941178	total: 32.5ms	remaining: 3.58s
9:	learn: 0.5893167	total: 36.2ms	remaining: 3.59s
10:	learn: 0.5856666	total: 39.8ms	remaining: 3.58s
11:	learn: 0.5821017	total: 43.1ms	remaining: 3.55s
12:	learn: 0.5786853	total: 46.4ms	remaining: 3.52s
13:	learn: 0.5756694	total: 49.8ms	remaining: 3.51s
14:	learn: 0.5733002	total: 53.4ms	remaining: 3.5s
15:	learn: 0.5706374	total: 57ms	remaining: 3.5s
16:	learn: 0.5683691	total: 60.2ms	remaining: 3.48s
17:	learn: 0.5667672	total: 63.7ms	remaining: 3.48s
18:	learn: 0.5653223	total: 66.6ms	remaining: 3.44

2026/04/12 11:20:47 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


982:	learn: 0.4731129	total: 3.4s	remaining: 58.9ms
983:	learn: 0.4730574	total: 3.41s	remaining: 55.4ms
984:	learn: 0.4730315	total: 3.41s	remaining: 51.9ms
985:	learn: 0.4729988	total: 3.42s	remaining: 48.5ms
986:	learn: 0.4729595	total: 3.42s	remaining: 45.1ms
987:	learn: 0.4729184	total: 3.43s	remaining: 41.6ms
988:	learn: 0.4728849	total: 3.43s	remaining: 38.2ms
989:	learn: 0.4728505	total: 3.43s	remaining: 34.7ms
990:	learn: 0.4728105	total: 3.44s	remaining: 31.2ms
991:	learn: 0.4727905	total: 3.44s	remaining: 27.7ms
992:	learn: 0.4727780	total: 3.44s	remaining: 24.3ms
993:	learn: 0.4727527	total: 3.45s	remaining: 20.8ms
994:	learn: 0.4727179	total: 3.45s	remaining: 17.3ms
995:	learn: 0.4727011	total: 3.45s	remaining: 13.9ms
996:	learn: 0.4726649	total: 3.46s	remaining: 10.4ms
997:	learn: 0.4726213	total: 3.46s	remaining: 6.93ms
998:	learn: 0.4725722	total: 3.46s	remaining: 3.47ms
999:	learn: 0.4725412	total: 3.47s	remaining: 0us


2026/04/12 11:20:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/12 11:21:02 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. 

🏃 View run Catboost at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2/runs/59ace58e62d74bc4909abfbd80850601
🧪 View experiment at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2


### Ensamble 6: Voting

In [49]:
with mlflow.start_run(run_name = "Ensamble de Votos") as run:

    
    algorithm1 = CatBoostClassifier()
    algorithm2 = RandomForestClassifier()
    algorithm3 = LGBMClassifier()

    voting_clf = VotingClassifier(
        estimators = [
            ("catboost", algorithm1),
            ("rf", algorithm2),
            ("lgbm", algorithm3),
        ],
        voting = "hard"
    )
    
    pipeline = Pipeline(
        steps = [
            ("imputer", SimpleImputer(strategy = "mean")),
            ("voting", voting_clf)
        ]
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    _accuracy_score = accuracy_score(y_test, predictions)
    _f1_score = f1_score(y_test, predictions)
    
    mlflow.log_metrics(
        {
            "accuracy": _accuracy_score,
            "f1": _f1_score
        }
    )

2026/04/12 11:21:11 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Learning rate set to 0.070218
0:	learn: 0.6738107	total: 4.75ms	remaining: 4.75s
1:	learn: 0.6577171	total: 8.26ms	remaining: 4.12s
2:	learn: 0.6435684	total: 11.7ms	remaining: 3.9s
3:	learn: 0.6321459	total: 15.1ms	remaining: 3.76s
4:	learn: 0.6222569	total: 18.4ms	remaining: 3.65s
5:	learn: 0.6135038	total: 22ms	remaining: 3.64s
6:	learn: 0.6061515	total: 25.5ms	remaining: 3.62s
7:	learn: 0.5998937	total: 29.1ms	remaining: 3.61s
8:	learn: 0.5941178	total: 33.1ms	remaining: 3.64s
9:	learn: 0.5893167	total: 36.7ms	remaining: 3.64s
10:	learn: 0.5856666	total: 40.3ms	remaining: 3.62s
11:	learn: 0.5821017	total: 43.3ms	remaining: 3.57s
12:	learn: 0.5786853	total: 47ms	remaining: 3.57s
13:	learn: 0.5756694	total: 50.5ms	remaining: 3.56s
14:	learn: 0.5733002	total: 54.6ms	remaining: 3.59s
15:	learn: 0.5706374	total: 58.5ms	remaining: 3.6s
16:	learn: 0.5683691	total: 61.8ms	remaining: 3.58s
17:	learn: 0.5667672	total: 65.6ms	remaining: 3.58s
18:	learn: 0.5653223	total: 68.9ms	remaining: 3.56

2026/04/12 11:21:20 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/12 11:21:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution

🏃 View run Ensamble de Votos at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2/runs/0476d317791f4b02a443543b030fd607
🧪 View experiment at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2


### Stacking

In [50]:
with mlflow.start_run(run_name = "Ensamble de Pilas") as run:

    
    algorithm1 = CatBoostClassifier()
    algorithm2 = XGBClassifier()
    algorithm3 = LGBMClassifier()

    voting_clf = StackingClassifier(
        estimators = [
            ("catboost", algorithm1),
            ("xgb", algorithm2),
            ("lgbm", algorithm3),
        ],
        final_estimator = RandomForestClassifier()
    )
    
    pipeline = Pipeline(
        steps = [
            ("imputer", SimpleImputer(strategy = "mean")),
            ("stacking", voting_clf)
        ]
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    _accuracy_score = accuracy_score(y_test, predictions)
    _f1_score = f1_score(y_test, predictions)
    
    mlflow.log_metrics(
        {
            "accuracy": _accuracy_score,
            "f1": _f1_score
        }
    )

2026/04/12 11:32:09 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Learning rate set to 0.070218
0:	learn: 0.6738107	total: 4.71ms	remaining: 4.7s
1:	learn: 0.6577171	total: 7.61ms	remaining: 3.79s
2:	learn: 0.6435684	total: 11ms	remaining: 3.65s
3:	learn: 0.6321459	total: 14.2ms	remaining: 3.54s
4:	learn: 0.6222569	total: 17.3ms	remaining: 3.44s
5:	learn: 0.6135038	total: 20.9ms	remaining: 3.46s
6:	learn: 0.6061515	total: 24.1ms	remaining: 3.42s
7:	learn: 0.5998937	total: 27.5ms	remaining: 3.4s
8:	learn: 0.5941178	total: 30.6ms	remaining: 3.37s
9:	learn: 0.5893167	total: 34.2ms	remaining: 3.39s
10:	learn: 0.5856666	total: 37.2ms	remaining: 3.35s
11:	learn: 0.5821017	total: 40.2ms	remaining: 3.31s
12:	learn: 0.5786853	total: 43.3ms	remaining: 3.29s
13:	learn: 0.5756694	total: 46.7ms	remaining: 3.29s
14:	learn: 0.5733002	total: 50.3ms	remaining: 3.3s
15:	learn: 0.5706374	total: 53.7ms	remaining: 3.3s
16:	learn: 0.5683691	total: 57ms	remaining: 3.29s
17:	learn: 0.5667672	total: 60.4ms	remaining: 3.3s
18:	learn: 0.5653223	total: 63.3ms	remaining: 3.27s
1

2026/04/12 11:32:42 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/opt/miniconda3/envs/ciencia-de-datos/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/12 11:32:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution

🏃 View run Ensamble de Pilas at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2/runs/6e368bf3784a4a91bcb695dec009025c
🧪 View experiment at: https://dagshub.com/davidrodriguez2712/ml-2-modulo.mlflow/#/experiments/2


In [33]:
mlflow.create_experiment("DSRP - Booking Clase 14 Julio - Con mas variables")

'2'

In [34]:
mlflow.set_experiment("DSRP - Booking Clase 14 Julio - Con mas variables")

<Experiment: artifact_location='mlflow-artifacts:/00d5fb7c83b14b4d8e2727d8b811cfa1', creation_time=1776009501288, experiment_id='2', last_update_time=1776009501288, lifecycle_stage='active', name='DSRP - Booking Clase 14 Julio - Con mas variables', tags={}, trace_location=None, workspace='default'>